# Phụ lục: Nguồn số liệu và tái lập

## Bộ MS1 đã chốt

Nhóm xác nhận các thư mục sau là bộ bàn giao cuối. Bản sao số liệu kiểm định trong repository cho phép build lại báo cáo mà không gọi Drive hoặc crawl lại. Các tệp gốc không bị sửa; SHA-256 và phương pháp trích xuất được giữ trong provenance.

In [ ]:
from pathlib import Path
import sys,json,hashlib
from collections import Counter
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display,HTML,Markdown
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'_config.yml').exists())
sys.path.insert(0,str(ROOT/'utils'))
from preliminary_results import load_members,corpus_metrics,session_members,run_range,paper_timings
folder=ROOT/'summaries/milestone1/frozen-20261127'
receipt=json.loads((folder/'provenance.json').read_text(encoding='utf-8'))
raw=(folder/'corpus.json').read_bytes()
assert hashlib.sha256(raw).hexdigest()==receipt['public_sha256']
papers=json.loads(raw)['papers']
assert len(papers)==25000 and len({p['paper_id'] for p in papers})==25000
members=load_members(ROOT)
sessions=session_members(ROOT,members)
measurement=corpus_metrics(members)[-1]
def table(rows):
    frame=pd.DataFrame(rows,dtype=object).where(lambda f:f.notna(),'Không áp dụng')
    def fmt(v):
        if isinstance(v,int) and not isinstance(v,bool):return f'{v:,}'.replace(',','.')
        if isinstance(v,float):return f'{v:,.2f}'.replace(',','_').replace('.',',').replace('_','.')
        return str(v)
    display(HTML(frame.map(fmt).to_html(index=False,escape=True,border=0)))


In [ ]:
table([{'Dải dữ liệu':i+1,'Thư mục đã chốt':'https://drive.google.com/drive/folders/'+receipt['source_folders'][src['member']],
        'Tệp gốc':src['original_name'],'Thời điểm dữ liệu (UTC)':src['generated_at'],
        'SHA-256 gốc':src['original_sha256']} for i,src in enumerate(receipt['sources'])])

## Quy tắc tổng hợp

Số paper, version, source và references của bộ cuối được đọc từ chi tiết từng paper trong audit hoặc validation của bản kiểm kê bàn giao. Các số tổng không khớp chi tiết không được dùng thay cho dữ liệu từng paper. Mỗi định danh chỉ được tính một lần. References là số quan hệ đã đọc được theo validator, không phải tập tài liệu duy nhất toàn cục.

Phép đo dung lượng và metadata references dùng các bản cumulative trước đó trên năm dải, có mẫu số riêng. Đây là số liệu thực nghiệm được giữ theo phạm vi đo, không phải một lần xuất lại toàn bộ bộ chốt.

In [ ]:
table([{'Dải phép đo':i+1,'Dải ID':m['assigned_range'],'Ngày xuất cumulative (UTC)':m['cumulative']['generated_at'],
        'Paper có cặp đo':m['cumulative']['sizes']['paired_papers']} for i,m in enumerate(members)])

## Cấu hình và định danh benchmark

Thông tin dưới đây xác định phiên thực thi, phiên bản code và môi trường. Xác nhận CPU-only là dữ liệu do notebook thu thập hoặc người chạy ghi; thông tin môi trường không được suy từ tốc độ chạy.

In [ ]:
table([{'Phiên đo':i+1,'Run ID':s['benchmark']['run_id'],
        'Source commit':s['benchmark']['environment'].get('source_commit'),
        'CPU logic':s['benchmark']['environment'].get('logical_cpu_count'),
        'CPU-only ghi nhận':s['benchmark']['environment'].get('cpu_only_confirmation'),
        'Bắt đầu (UTC)':s['benchmark']['started_at'],'Kết thúc (UTC)':s['benchmark']['finished_at']}
       for i,s in enumerate(sessions)])

## Biên dịch báo cáo

README của repository hướng dẫn cài Python, dependencies, kiểm thử và build. Jupyter Book bắt buộc thực thi notebook; các bảng, biểu đồ và CSV timing được sinh khi build, không dán ảnh chụp kết quả. Input code được ẩn khỏi HTML để bản nộp tập trung vào phương pháp và phân tích.

| Nguồn | Cách dùng |
|---|---|
| Bộ chốt và validation chi tiết | Kết quả thu thập, trạng thái chất lượng toàn nhóm |
| Cumulative và sidecar | Thực nghiệm kích thước, mức đầy đủ metadata references |
| Benchmark, samples, timings | Thời gian, RAM, disk của phiên đo |
| Provenance | Định danh nguồn, phạm vi và SHA-256 |

Đề bài: [Milestone 1 — Data Crawling](https://hcmus-cs-ta.github.io/lab-book/intro-ds/statement/crawling/).